# Estimate genotype-phenotype associations

**Purpose.** Estimate guide- or gene-level effects by modelling phenotype scores as a function of guide abundance.

**Recommended use.** Use after phenotype scores and matched guide-count tables have been generated for the same experimental wells.

**Primary outputs.** Effect estimates, uncertainty or resampling support, adjusted significance values, diagnostic plots, and ranked findings.

---

> Paths in this notebook are placeholders. Set `src` and any other required path to the experimental data before execution.
> spaCR writes outputs within, or immediately adjacent to, the configured source directory unless an explicit output path is set.

## 1. Verify the environment

The following cell reports the installed spaCR version. Import errors must be resolved before the analysis cells are executed. GPU acceleration is optional and depends on the selected workflow and installed computational backend.

In [ ]:
import spacr
from spacr.version import version_str

print(version_str)

## 2. API entry point

This workflow calls the following public function:

- [`spacr.ml.perform_regression`](https://einarolafsson.github.io/spacr/api/spacr/ml/index.html#spacr.ml.perform_regression)

```python
perform_regression(settings)
```

Parameter definitions and defaults are generated from the same public API and are listed in the settings reference below.

In [ ]:
from spacr.ml import perform_regression

## 3. Settings and API reference

Review the parameter definitions here, then edit the values in the categorized code cells below. Required settings must be supplied. Optional settings retain the displayed default when unchanged. Conditionally required settings are necessary only for the indicated analysis branch. Defaults and descriptions are generated from the installed spaCR version so that the notebook remains aligned with the public API.

### [`spacr.ml.perform_regression`](https://einarolafsson.github.io/spacr/api/spacr/ml/index.html#spacr.ml.perform_regression)


#### Paths

- **`metadata_files`** *(optional)* — (list) - Gene-annotation CSVs, each with a 'Gene ID' column, that are joined onto the regression results by gene, writing an extra results CSV per file. These are gene tables, not plate/well metadata. When toxo is True the order matters: index 0 is read as the ME49 transcription table and index 1 as the GT1 phenotype table. Default [].
- **`paired_data`** *(optional)* — (list of dicts) - Regression input table: each row explicitly pairs one score CSV with one count CSV. Plate identity comes from both files when they agree, from the partner when only one declares plateID, or from the row order when neither does. A conflict is refused. Legacy score_data/count_data lists are migrated positionally and logged. Default [].

#### Plate Layout & Controls

- **`level`** *(optional)* — (str) - Select the regression fit level or the proportion-summary unit. Regression: 'both' runs separate gRNA and gene fits, writes results_grna.csv and results_gene.csv, and corrects each fit independently with multiple_testing_method. This avoids a collinear combined design because a gene fraction is the sum of its guide fractions. 'grna' or 'gene' runs one fit. Disabled for mixed models, which nest guides within genes. Proportion plots: 'object' pools objects, 'well' averages by well, and 'plate' averages by plate. Default 'both' for regression and 'object' for proportions.
- **`positive_control`** *(optional)* — (str) - Identifier of the positive-control class. In ML screening it is the value in location_column (e.g. 'c2') whose objects are labelled class 1 for training; in gRNA regression it is a gene/gRNA ID substring (e.g. '239740') matched against coefficient names to tag them 'pc' in the results and volcano plot. Defaults 'c2' and '239740' respectively.
- **`negative_control`** *(optional)* — (str) - Identifier of the negative-control class. In ML screening it is the value in location_column (e.g. 'c1') whose objects are labelled class 0 for training; in gRNA regression it is a gene/gRNA ID substring (e.g. '233460') matched against coefficient names to tag them 'nc' in the results and volcano plot. Defaults 'c1' and '233460' respectively.
- **`exclude_grnas`** *(optional)* — (list or str) - gRNA or gene identifiers known not to occur in cells, such as primer or plasmid carry-over. These sequences are removed before guide fractions are calculated, and retained guides are renormalized. This differs from background subtraction, which corrects spurious reads assigned to a real guide. A gene identifier matches all of its guides. Default None.
- **`positive_control_wells`** *(optional)* — (list or str) - Wells containing only the positive control, e.g. ['c2']. Accepts rows (r1), columns (c1), or individual wells (A01); the Plate button selects them from a map. Pure controls are calibration references rather than screen observations, so they are excluded from the regression. They define the positive endpoint for mixed-ratio calibration and should be identified from the plate design. Default None.
- **`negative_control_wells`** *(optional)* — (list or str) - Wells containing only the negative control, e.g. ['c1']. Accepts the same row, column, and well notation as positive_control_wells. These wells are excluded from the regression and define the negative endpoint for mixed-ratio calibration. Default None.
- **`mixed_control_wells`** *(optional)* — (list or str) - Wells holding a known mixture of the positive and negative controls, e.g. ['c3']. Removed from the regression like the other two. These provide strong validation because per-cell identities are unknown while the aggregate proportions are known from sequencing, allowing annotation methods to be scored on real rather than simulated cells. Default None.
- **`controls`** *(optional)* — (list) - gRNA identifiers treated as non-targeting controls. Their coefficients define the effect-size cutoff drawn on the volcano plot: abs(median(control coefficients)) + threshold_multiplier × spread, where threshold_method selects the spread estimator. A wider control distribution raises the cutoff. None disables the effect-size cutoff. Default ['000000'] -- the non-cutting control GENE, which spaCR resolves to every one of its guides in the library you loaded, rather than a hand-typed list that goes stale. A guide id works too, with or without the organism prefix.
- **`filter_column`** *(optional)* — (str) - Metadata column used to drop control wells before regression: every row whose value appears in filter_value is removed from both the score data and the read counts. Use 'columnID' (default) when controls sit in plate columns, 'rowID' when they sit in rows. In annotate_filter_vision it instead names the score column thresholded by upper_threshold/lower_threshold.
- **`filter_value`** *(optional)* — (list) - Values of filter_column whose rows are removed - not kept - before regression, normally the control columns; default ['c1','c2','c3']. Dropping them stops control wells from dominating the gene and gRNA fits. Only list values take effect: a bare string is silently ignored and nothing is filtered.
- **`batch_correction`** *(optional)* — (str) - Plate/batch correction applied before Image UMAP, ML screen classification or phenotype regression. 'none' leaves measurements alone; 'center' removes each plate's mean shift; 'zscore' aligns plate means and variances; 'robust_zscore' uses median/MAD and tolerates outliers; 'combat' models the batch effect while protecting the terms named in batch_covariate_column. Correct when plates were stained or imaged separately; leave off when they were not, since every method removes real signal that happens to align with plate. See spacr.batch_correction.correct_batch_effects. Default 'none'.
- **`batch_column`** *(optional)* — (str) - Metadata column that identifies independent acquisition batches, normally 'plateID'. Every analyzed row must have a value and at least batch_min_samples rows must occur in each batch. Use an acquisition date or instrument ID only if that is the nuisance source you intend to remove. Default 'plateID'. API: spacr.batch_correction.correct_batch_effects.
- **`batch_control_column`** *(optional)* — (str or None) - Metadata column containing reference-control labels for control_center, normally 'columnID' for plate controls. It is ignored by center, zscore, robust_zscore, and none. Blank follows col_to_compare in Image UMAP or location_column in Classify (ML); regression defaults to 'columnID'. API: spacr.batch_correction.correct_batch_effects.
- **`batch_control_values`** *(optional)* — (str, number, list or None) - Reference/negative-control value(s) in batch_control_column used by control_center. Each plate needs at least batch_min_samples matching rows. Image UMAP falls back to neg and Classify (ML) to negative_control when this field is blank; regression requires an explicit value. Default varies by module. API: spacr.batch_correction.correct_batch_effects.
- **`batch_covariate_column`** *(optional)* — (str, list or None) - Metadata column(s) naming the biology combat must PROTECT, e.g. 'condition' or 'condition,timepoint'. combat estimates the batch effect from the residuals after these terms, so anything NOT listed is treated as noise and removed along with the plate effect. Leave your treatment out of this list and combat will quietly delete the effect you are measuring. See spacr.batch_correction.correct_batch_effects. Default None.
- **`batch_combat_mean_only`** *(optional)* — (bool) - True corrects only the additive batch shift and leaves each batch's scale alone. Use it when the plates differ in level but not in spread, or when a batch has too few rows for a stable variance estimate. False (the default) corrects both location and scale, which is standard ComBat. Ignored by every method other than combat. API: spacr.batch_correction.correct_batch_effects.
- **`batch_min_samples`** *(optional)* — (int) - Minimum number of rows required in every batch, and minimum matching reference controls per batch for control_center. Correction stops with an actionable error below this threshold because a one- or two-object plate estimate is unstable. Default 3. API: spacr.batch_correction.correct_batch_effects.
- **`batch_missing_control`** *(optional)* — (str) - Policy when control_center cannot find enough reference controls on a plate: 'error' stops rather than silently mixing corrected and raw plates; 'skip' leaves that plate unchanged and records a warning. Default 'error'. API: spacr.batch_correction.correct_batch_effects.

#### Regression: Response

- **`dependent_variable`** *(optional)* — (str) - Name of the column in score_data that is modelled as the response, e.g. 'pred'/'predictions' from the ML scoring step or a measured feature such as 'pathogen_nucleus_shortest_distance'. It is aggregated per well by agg_type and then optionally transformed. The run aborts if the column is absent from the score CSV. Default 'pred'.
- **`invert_dependent_variable`** *(optional)* — (bool or int) - Flip the response before it is aggregated per well, for scores whose useful direction is downward. False or 0 leaves it as measured, True or 1 uses 1 - x (right for a probability, so a low infection score becomes a high phenotype), and -1 uses 1 / x (right for a distance or a count). Any other value raises ValueError in process_scores. It changes the sign of every coefficient and therefore which side of the volcano your hits land on. Default False.
- **`count_grna_column`** *(optional)* — (str) - Name of the column in the count CSV holding the guide identifier. It was hard-coded to 'grna', so a file naming it 'sgRNA' or 'guide' died on a message listing the four columns spaCR wanted and none of the ones the file actually had. Set it to whatever your sequencing pipeline wrote. Default 'grna'.
- **`count_value_column`** *(optional)* — (str) - Name of the column in the count CSV holding the read count for one guide in one well; it becomes the per-well fraction the fraction_threshold sweep works on. Hard-coded to 'count' until now, so a file naming it 'reads' or 'n' failed with a message naming only the columns spaCR expected. Default 'count'.
- **`analysis_unit`** *(optional)* — (str) - What one row of the model is. 'well' collapses each well's objects into a single value with agg_type first, so the well is the independent unit and the number of cells behind it only affects precision. 'cell' regresses the individual objects instead, which keeps power but treats cells from one well as independent when they are not, so standard errors are optimistic unless the model accounts for the clustering (regression_type='mixed'). This is the explicit spelling of agg_type=None, which used to change the unit of analysis silently. Default 'well'.
- **`agg_type`** *(optional)* — (str) - How per-object scores are collapsed to one value per well before regression: 'mean', 'median', 'quantile' (75th percentile), or None to skip aggregation and regress on individual objects. Median resists a handful of extreme cells; None keeps power but ignores within-well correlation. Forced to a per-well sum for poisson and to None for quantile. Default 'mean'.
- **`transform`** *(optional)* — (str) - Optional transform applied to the aggregated per-well response before fitting: 'log' (log1p), 'sqrt', 'square', 'beta' (logit, for a response that is a proportion - the endpoints are squeezed off 0 and 1 first and the summary says so), or None for none. Reach for it when the response is skewed and the normality check fails; the fit then reports coefficients for the transformed column, named '&lt;transform&gt;_&lt;dependent_variable&gt;'. Default None.
- **`glm_transform_conflict`** *(optional)* — (str) - Response scale to fit when transform is itself a link ('log' or 'logit') and the automatically selected GLM family also has a non-identity link. 'untransformed' fits the measured response and lets the family link transform it; use this for a proportion that was unnecessarily transformed first. 'transformed' keeps the transformed response and fits a Gaussian identity-link model. 'warn' reproduces the legacy double-transform behavior and prints a warning. This setting has no effect for other transforms or regression types. Default 'untransformed'.

#### Regression: Model

- **`inference`** *(optional)* — (str) - How effects are tested; the readable front end for analysis_mode. Default 'nonparametric': each guide is a plate-blocked Freedman-Lane permutation with an empirical P, valid however many guides there are, but no P can be below 1/(guide_permutations + 1). 'parametric' fits every guide at once in the chosen regression_type, so it needs more wells than guides or no coefficient is identifiable. 'auto' counts guides and wells and takes the simultaneous fit only when the design supports it.
- **`analysis_mode`** *(optional)* — (str) - 'regression' fits the selected simultaneous model. 'guide_permutation' tests each guide as a plate-adjusted marginal association using blocked Freedman--Lane permutations and then corrects the requested support family. Normally set for you by 'inference'; set it directly only to override that choice. Default 'regression'.
- **`regression_type`** *(optional)* — (str) - Model family. Default 'mixed' nests guides inside genes as random effects, so guides disagreeing widens that gene's interval; it answers both levels at once, which is why 'level' greys out. Otherwise: 'ols', 'wls', 'rlm', 'huber' or 'quantile' for a continuous response; 'logit', 'probit', 'beta' or 'quasi_binomial' for fractions; 'poisson' for counts; 'lasso', 'ridge', 'elasticnet' or 'horseshoe' when the predictors outnumber the wells. Note regression_type 'beta' fits a beta GLM, while transform 'beta' only transforms the response.
- **`regression_backend`** *(optional)* — (str) - Selects the library and device that fit the chosen regression_type. 'statsmodels (CPU)' preserves the established results and is the default. 'torch (GPU)' accelerates mixed models, 'pyfixest (CPU)' accelerates OLS/WLS with absorbed plate-position effects, and 'glum (CPU)' accelerates wide GLMs. The selector describes measured costs and numerical differences for each option; unavailable or incompatible backends are greyed out with the reason. Default 'statsmodels (CPU)'.
- **`model_plate_position`** *(optional)* — (bool) - Include plate row and column (rowID, columnID) in the model. random_row_column_effects then chooses fixed or random terms. Turning this off while random effects are on is refused because there is nothing left to make random. In a 610-well screen, the 35 position terms were jointly significant (p=6.7e-23); omitting them increased residual SD 7.2% and changed the hit list. With no position effect, including them increased standard errors 1.6%. Default False; enable it when edge, row, or column effects are plausible.
- **`random_row_column_effects`** *(optional)* — (bool) - Fit plate, row and column as random effects instead of fixed ones: True overrides regression_type to 'mixed' and fits a MixedLM grouped by plateID with rowID and columnID variance components, dropping them from the fixed-effect formula. Use it when edge or row artefacts differ between plates; it is slower and may fail to converge. Default False.
- **`cov_type`** *(optional)* — (str) - Heteroscedasticity-robust covariance estimator passed to the likelihood fits: 'HC0', 'HC1', 'HC2' or 'HC3', or None for classical non-robust errors. It changes standard errors and p-values only, never the coefficients; reach for 'HC3' when residual variance grows with well cell count. The penalised, robust and quantile fits have no such estimator and refuse it rather than quietly reporting ordinary errors under a robust label. Default None. Read by regression_type 'glm', 'logit', 'ols', 'poisson', 'probit', 'quasi_binomial', 'wls'.

#### Regression: Model Tuning

- **`alpha`** *(optional)* — (float) - Regularisation strength for the penalised models only: the L1 penalty for 'lasso', the L2 penalty for 'ridge', the combined penalty for 'elasticnet' and the inverse margin for 'hinge'. Larger values shrink more coefficients toward zero; set it to 'auto' or None to choose it by 5-fold cross-validation, which is usually what you want because the default 1 shrinks a fraction-scale design to nothing. Every other family refuses a non-default alpha rather than ignoring it. Default 1. Read by regression_type 'elasticnet', 'hinge', 'lasso', 'ridge'.
- **`l1_ratio`** *(optional)* — (float) - How the elastic-net penalty is split between L1 and L2: 1.0 is a pure lasso (sparse, picks one gRNA out of a correlated group), 0.0 is a pure ridge (dense, shares the effect across the group), and values between keep some of both. Use 0.5 when correlated gRNAs of the same gene should be selected together rather than arbitrarily. Read only by regression_type 'elasticnet'. Default 0.5.
- **`quantile`** *(optional)* — (float) - Which quantile of the response quantile regression fits, strictly inside 0 and 1: 0.5 is the median (robust to outlier wells), 0.9 asks which gRNAs move the top of the distribution rather than its centre. Aggregation is turned off automatically so the quantile is taken over cells, not over well means. Read only by regression_type 'quantile'; it replaced the old overload of alpha. Default 0.5.
- **`huber_t`** *(optional)* — (float) - Where Huber's loss switches from squared to linear, in units of the estimated residual scale, for the robust fits. Smaller values downweight more wells and resist heavier contamination; larger values approach ordinary least squares. The default 1.345 gives 95 percent of the efficiency of OLS when the residuals really are normal. Read only by regression_type 'rlm' and 'huber'. Default 1.345.
- **`hinge_threshold`** *(optional)* — (float) - Response value above which a well counts as positive for the hinge (linear SVM) fit. Leave it None when the response is already binary, in which case the two values it holds become the two classes. spaCR refuses a continuous response with no threshold rather than splitting it at the mean or median, because a cut chosen by the software decides the hypothesis being tested. Read only by regression_type 'hinge'. Default None.
- **`hinge_n_boot`** *(optional)* — (int) - Number of bootstrap resamples behind the hinge p-values. A support vector machine has no likelihood and so no Wald test; spaCR refits it on this many resamples of the wells and compares each coefficient to its bootstrap standard deviation. Treat the result as a stability statistic, not a hypothesis test. Higher is steadier and linearly slower; below about 50 the standard deviations are too noisy to rank on. Default 200. Read by regression_type 'hinge'.
- **`lasso_n_boot`** *(optional)* — (int) - Number of bootstrap resamples used to rank lasso and elastic-net hits by how often each gRNA survives the penalty. These models have no valid p-values, so selection frequency replaces the significance test entirely. Higher is steadier and linearly slower; the cost is one full penalised fit per resample, doubled when alpha is 'auto' because each resample cross-validates. Default 200. Read by regression_type 'elasticnet', 'group_lasso', 'lasso'.
- **`lasso_selection_threshold`** *(optional)* — (float) - Minimum bootstrap selection frequency, between 0 and 1, for a lasso or elastic-net coefficient to be called a hit. 0.6 means the gRNA kept a non-zero coefficient in at least three fifths of the resamples. Raise it for a shorter, harder-to-argue-with list; lowering it below about 0.5 admits terms the penalty drops as often as it keeps. Default 0.6. Read by regression_type 'elasticnet', 'group_lasso', 'lasso'.
- **`group_lasso_lambda`** *(optional)* — (float) - Penalty weight of the group lasso, which shrinks all of one gene's guides together rather than one at a time, so a gene enters or leaves the model as a unit instead of on its luckiest guide. Larger values keep fewer genes; 0 leaves the fit unpenalised and negative is refused. Default 0.05. Read by regression_type 'group_lasso'.

#### Regression: Permutation Test

- **`grna_statistic`** *(optional)* — (str) - What the permutation test measures between a gRNA's well fractions and the well phenotype. 'pearson' is a partial correlation, which is linear and is moved by an extreme well in proportion to how extreme it is. 'rank' is the same quantity computed on the ranked phenotype, so it responds to order rather than magnitude and no single well can move it far. Both cost one matrix product, so the choice does not change how long the test takes. Default 'pearson'.
- **`guide_min_wells`** *(optional)* — (int or list) - Minimum numbers of independent wells containing a guide. A list such as [1, 2, 3, 4] writes one sensitivity-analysis table and volcano plot per threshold; P values are computed once and the multiple-testing correction is repeated within each eligible family. Default [1, 2, 3, 4].
- **`guide_primary_min_wells`** *(optional)* — (int or None) - Which guide_min_wells family supplies results_significant.csv and the returned 'significant' table. Default None chooses the smallest requested threshold.
- **`guide_permutations`** *(optional)* — (int) - Number of plate-blocked Freedman--Lane residual permutations used for empirical two-sided guide P values. The P value IS (exceedances + 1) / (permutations + 1), where an exceedance is a permuted statistic at least as extreme as the observed one -- so the smallest it can be is 1 / (permutations + 1), reached only when nothing exceeded: 1,000 permutations resolve to about 1e-3, 10,000 to 1e-4, and the default 200,000 to 5e-6. Raise this when the volcano shows a flat row of guides at the P-value floor; runtime increases linearly. Default 200000.
- **`guide_permutation_seed`** *(optional)* — (int) - Random seed for reproducible residual permutations. Keep it fixed to reproduce exact empirical P values; change it to check Monte Carlo sensitivity. Default 0.
- **`guide_permutation_block`** *(optional)* — (str) - Column defining exchangeability blocks for permutations, normally plateID. Residuals are never shuffled between its levels. Default 'plateID'.
- **`guide_nuisance_columns`** *(optional)* — (list) - Additional measured well-level covariates to residualize from both phenotype and guide fraction before testing. Do not put post-treatment outcomes here. Default [].
- **`guide_presence_threshold`** *(optional)* — (float) - A guide counts as present in a well only when its fraction is above this value. The effect still uses the unthresholded fraction. Default 0.0.
- **`guide_permutation_batch_size`** *(optional)* — (int) - Number of permutation outcomes evaluated together. Lower this if memory is tight; it does not change the result. Default 500.

#### Regression: Significance

- **`multiple_testing_method`** *(optional)* — (str) - Correction applied within each outcome/support family: fdr_bh (Benjamini--Hochberg, default), fdr_by, bonferroni, holm, or none. Stricter family-wise methods generally call fewer guides.
- **`fdr_alpha`** *(optional)* — (float) - Family-level rejection threshold for adjusted P values in guide_permutation mode. Must be between 0 and 1. Default 0.05.
- **`threshold_method`** *(optional)* — (str) - Select the spread estimator for the control-based effect-size cutoff: 'std', legacy 'var' (squared units), 'mad', 'iqr', 'percentile' (the 95th percentile of absolute coefficients), or 'range'. 'none' disables the effect-size cutoff. Historical aliases such as 'standard_deveation', 'variance', and 'quantile' are accepted. Used only when controls are set. Default 'std'.
- **`threshold_multiplier`** *(optional)* — (float) - Set how many control-distribution spreads are required for a hit. The cutoff is abs(median(control coefficients)) + threshold_multiplier × spread, using threshold_method for the spread. Larger values demand a larger effect; threshold_method='none' disables the cutoff. Used only when controls are set. Default 3.
- **`Toxoplasma`** *(optional)* — (bool) - Join the bundled Toxoplasma annotation onto every exported table and colour the volcano by it: gene name and product, signal peptide and transmembrane domain from the project's DeepTMHMM run, hyperLOPIT/TAGM compartment, the published CRISPR fitness scores, and tachyzoite / tissue-cyst / EES1-5 expression. Joined on the gene NUMBER, so TGGT1 and TGME49 ids meet. Writes supplementary_topology.csv beside the results. Turn it off for non-Toxoplasma screens. Was called 'toxo' before 2026-08-17; an old settings CSV still loads. Default True.
- **`p_threshold_alpha`** *(optional)* — (float) - The P value a coefficient has to beat to be called a hit, and the line the volcano draws. It cuts on whichever P p_threshold_kind names, so results_significant.csv and the figure printed beside it agree: until this existed the plot's own right-click raw/adjusted choice was the only say anyone had, and the table and the picture could mean two different things by 'significant'. A fraction, strictly between 0 and 1 -- 5 for '5%' is refused. Default 0.05.
- **`p_threshold_kind`** *(optional)* — (str) - Whether p_threshold_alpha cuts on the ADJUSTED P value ('adjusted' -- the multiple-testing-corrected column multiple_testing_method produces) or on the raw per-coefficient P ('raw'). The volcano's right-click menu offers the same two, and this is the one the RUN uses, so the exported hits and the plot cannot disagree. 'raw' calls far more genes on a screen of thousands of guides. Anything else is refused rather than falling back. Default 'adjusted'.
- **`rra_alpha`** *(optional)* — (float) - The top fraction of the ranked guide list robust rank aggregation scores against: 0.25 asks whether a gene's guides cluster in the best quarter of the ranking more than chance allows, ignoring the rest. Smaller is stricter and returns fewer, better-supported genes. Above 0 and at most 1; 25 for '25%' is refused. Default 0.25. Read by regression_type 'rra'.
- **`rra_permutations`** *(optional)* — (int) - How many permuted rankings the robust rank aggregation null is built from. The smallest P value it can report is about 1/rra_permutations, so 10000 resolves the tail to 1e-4; raise it when many genes pile up at that floor and lower it while exploring, since the cost is linear in this number. Default 10000. Read by regression_type 'rra'.

#### Regression: Quality Filters

- **`min_cell_count`** *(optional)* — (int) - Wells with fewer than this many scored objects are dropped before regression. Raising it removes noisy, sparsely imaged wells at the cost of statistical power. Set it to None and spaCR simulates the count at which a well's mean score stabilises within tolerance and uses that value instead. Default 100.
- **`min_n`** *(optional)* — (int) - Observation count a significant hit must strictly exceed to appear in results_significant_filtered.csv: gRNA hits need n_grna &gt; min_n, gene hits need n_gene &gt; min_n. The unfiltered hit list is still written alongside it. Raise it to drop hits resting on one or two wells. Default 0, which filters nothing.
- **`fraction_threshold`** *(optional)* — (float) - Minimum relative abundance, 0-1, that a gRNA must reach within a well's total read count to be kept. Raising it strips low-abundance and bleed-through gRNAs and lowers the mean gRNAs per well; set it too high and every row is removed and the run errors out. Leave None to auto-pick the cutoff giving target_unique_count gRNAs per well. Default None.
- **`normalise_fraction`** *(optional)* — (bool) - Divide a gRNA's fraction by the sum of the fractions that remain in its well after fraction_threshold, before deciding how many cells it is given. On, a gRNA's share is measured against what survived the threshold; off, it is measured against every read the well produced, including those the threshold removed. The two differ whenever the threshold removes anything: normalising raises every surviving share, and by more the more was removed. Default True.
- **`target_unique_count`** *(optional)* — (int) - Desired mean number of distinct gRNAs per well. spaCR sweeps 1000 read-fraction thresholds, picks the one whose per-well mean unique gRNA count lands closest to this number, then discards every gRNA call below that fraction. Lower it for a stricter, cleaner well assignment; raise it to keep more gRNAs per well. Default 5.
- **`tolerance`** *(optional)* — (int or float) - How close a subsampled well mean has to be to the full-well mean before minimum_cell_simulation calls that sample size sufficient, which is what sets min_cell_count when you leave it None. An int is read as a percentage (2 means 2%), a float as a fraction (0.02 means the same); anything else raises ValueError. Tighten it toward 0.01 to demand more cells per well and drop more wells, loosen it to 0.05 to keep sparse wells at the cost of noisier per-well scores. Default 0.02.
- **`outlier_detection`** *(optional)* — (bool) - After building the regression table, drop gRNAs whose well count falls outside 1.5x the 5th-95th percentile spread, then recompute the per-gRNA tables. This removes gRNAs present in implausibly few or many wells that would otherwise dominate coefficients; disable it if your library is deliberately uneven. Default True.

#### Regression: Diagnostics

- **`regression_qc`** *(optional)* — (bool) - Write the regression QC suite -- variance homogeneity, residual, design, influence and calibration panels -- into &lt;res_folder&gt;/regression_qc/ as figures, a combined PDF and a text report. Roughly 5.8 seconds and 19 files per fit: right for one analysis, which is why it is on by default. A sweep turns it off on its own, since a hundred trials is ten minutes and two thousand files nobody opens; reopen a single trial to get its diagnostics back. Applies to every regression_type, because any of them can be badly specified. Default True.

#### Invasion Assay

- **`control_wells`** *(optional)* — (list or None) - Wells whose parasites carry no pre-permeabilisation stain, giving the honest negative distribution the cut should sit above -- better evidence than any automatic method. Name a column ('c12'), a row ('r1'), a well ('r1_c12') or a full plate key. These wells are dropped from every efficiency, since a staining control is not an experimental condition. None runs the automatic per-field method instead. NOTE the screen regression reads this same key for a different job, where it must be a list matching filter_value. Default None.

#### Advanced

- **`strict_errors`** *(optional)* — (bool or None) - What happens when a step hits a problem it could survive. OFF records the failure in the run ledger and the end-of-run summary and carries on with the items that worked. ON raises immediately on a setup or configuration error -- an unreadable path, a missing column, a database that will not open -- so a batch stops at the first sign its inputs are wrong instead of producing a plausible partial result. Per-item failures such as one corrupt image are survived either way. None defers to $SPACR_STRICT_ERRORS, which is how a cluster sets it for a whole batch. Default None.
- **`max_failure_rate`** *(optional)* — (float or None) - Fraction of failed items above which the run aborts rather than finishing and reporting. 0.2 means 'stop once more than a fifth of the fields have failed', on the grounds that whatever is left is no longer the experiment. The ledger is stamped into the artifact before the abort, so the evidence survives. None (the default) never aborts on rate alone - every failure is still counted and reported, and the artifact is still marked partial. Default None.
- **`verbose`** *(optional)* — (bool) - Print extra run detail instead of the minimal log: the resolved settings table, the channel and model choices per object type, per-table row counts, and how many objects survive each filter. It only adds console output, so turn it on when object counts come out unexpected and you need to see which stage removed them. The default differs per pipeline -- True for mask, UMAP, screen analysis, barcode mapping and Cellpose training; False for measure, the plotting helpers and regression.

## 4. Run

After editing the settings cell, run the function cell immediately below it. Long operations report progress through spaCR's logging system; set `SPACR_LOG_LEVEL=DEBUG` before starting Jupyter for more detail.

In [ ]:
settings = {
    # Paths
    # Optional settings
    'metadata_files': [],
    'paired_data': [],

    # Plate Layout & Controls
    # Optional settings
    'level': 'both',
    'positive_control': '239740',
    'negative_control': '233460',
    'exclude_grnas': None,
    'positive_control_wells': None,
    'negative_control_wells': None,
    'mixed_control_wells': None,
    'controls': ['000000'],
    'filter_column': 'columnID',
    'filter_value': ['c1', 'c2', 'c3'],
    'batch_correction': 'none',
    'batch_column': 'plateID',
    'batch_control_column': 'columnID',
    'batch_control_values': None,
    'batch_covariate_column': None,
    'batch_combat_mean_only': False,
    'batch_min_samples': 3,
    'batch_missing_control': 'error',

    # Regression: Response
    # Optional settings
    'dependent_variable': 'pred',
    'invert_dependent_variable': False,
    'count_grna_column': 'grna',
    'count_value_column': 'count',
    'analysis_unit': 'well',
    'agg_type': 'mean',
    'transform': 'log',
    'glm_transform_conflict': 'untransformed',

    # Regression: Model
    # Optional settings
    'inference': 'nonparametric',
    'analysis_mode': 'guide_permutation',
    'regression_type': 'mixed',
    'regression_backend': 'statsmodels (CPU)',
    'model_plate_position': False,
    'random_row_column_effects': False,
    'cov_type': None,

    # Regression: Model Tuning
    # Optional settings
    'alpha': 1,
    'l1_ratio': 0.5,
    'quantile': 0.5,
    'huber_t': 1.345,
    'hinge_threshold': None,
    'hinge_n_boot': 200,
    'lasso_n_boot': 200,
    'lasso_selection_threshold': 0.6,
    'group_lasso_lambda': 0.05,

    # Regression: Permutation Test
    # Optional settings
    'grna_statistic': 'pearson',
    'guide_min_wells': [1, 2, 3, 4],
    'guide_primary_min_wells': None,
    'guide_permutations': 200000,
    'guide_permutation_seed': 0,
    'guide_permutation_block': 'plateID',
    'guide_nuisance_columns': ['rowID', 'columnID'],
    'guide_presence_threshold': 0.0,
    'guide_permutation_batch_size': 500,

    # Regression: Significance
    # Optional settings
    'multiple_testing_method': 'fdr_bh',
    'fdr_alpha': 0.05,
    'threshold_method': 'std',
    'threshold_multiplier': 3,
    'Toxoplasma': True,
    'p_threshold_alpha': 0.05,
    'p_threshold_kind': 'adjusted',
    'rra_alpha': 0.25,
    'rra_permutations': 10000,

    # Regression: Quality Filters
    # Optional settings
    'min_cell_count': 100,
    'min_n': 0,
    'fraction_threshold': 0.02,
    'normalise_fraction': True,
    'target_unique_count': 5,
    'tolerance': 0.02,
    'outlier_detection': True,

    # Regression: Diagnostics
    # Optional settings
    'regression_qc': True,

    # Invasion Assay
    # Optional settings
    'control_wells': ['c1', 'c2', 'c3'],

    # Advanced
    # Optional settings
    'strict_errors': None,
    'max_failure_rate': None,
    'verbose': False,
}

In [ ]:
perform_regression(settings)

## Outputs and next steps

Effect estimates, uncertainty or resampling support, adjusted significance values, diagnostic plots, and ranked findings.

Output directories remain associated with the source dataset, which preserves plate-level provenance across subsequent spaCR workflows.

### Related documentation

- [Graphical workflow tutorials](https://einarolafsson.github.io/spacr/tutorials/)
- [Python API reference](https://einarolafsson.github.io/spacr/python_api.html)